# Parser field analysis — adjectives & neighbors

Two experiments that score the **two parse fields nothing else measures**.

`eval_parser_target_accuracy.py` scores the `target` field, because ScanRefer's
`object_name` is free ground truth for it. `adjectives` and `neighbors` have **no ground
truth at all** — and the manual annotation sheets meant to cover them were exported but
never filled in (0 of 200 rows, every parser). Yet those are the two fields the fusion
network actually consumes: TAF reads the attributes, A2F reads the adjacency.

| # | Script | Question |
|---|---|---|
| 1 | `parse_field_comparison.py` | How do the parsers differ on the two unscored fields? |
| 2 | `complex_sentence_showdown.py` | Where does the rule-based parser actually break down? |

Every table, chart and figure is rendered **inline in the cell output** as well as saved
under `outputs/analysis/`.

**No GPU. No checkpoint. No `predictions.p`.** Both read only the parse caches and
ScanRefer's JSON, and finish in seconds.

> **Scope.** These measure **parse quality, not grounding accuracy**. Whether a worse
> parse *causes* a worse box is a different experiment (`run_parse_corruption.py` →
> `parse_error_propagation.py`), which needs a GPU and has not been run. Do not quote
> anything here as causal evidence.

---
## 0 · Setup

Run **one** of the next two cells: local, or Colab.

### 0a · Local — just point at the repository

In [ ]:
import os

# EDIT THIS if the notebook is not already sitting in the repository root.
REPO = "/home/ario/PY/3D-VG"

os.chdir(REPO)
print("cwd:", os.getcwd())

### 0b · Colab — mount Drive and unzip instead

Skip this cell entirely when running locally. Only `data/` and `data_parsing/` are
needed — not `cached_scenes/`, not `outputs/`, because nothing here runs a model.

In [ ]:
# COLAB ONLY -- skip when running locally.
RUN_COLAB_SETUP = False

if RUN_COLAB_SETUP:
    from google.colab import drive
    drive.mount('/content/drive')

    # Adjust to wherever the repository lives in your Drive.
    %cd /content/drive/MyDrive/3D-VG

    !pip install -q spacy matplotlib
    !python -m spacy download en_core_web_sm

    import os
    print("cwd:", os.getcwd())

### 0c · Preflight — check the inputs exist before running anything

This cell only looks; it changes nothing. A missing parse cache is the one thing that
makes the experiments below fail, so it is worth ten seconds.

In [ ]:
import json, os, sys

PARSERS = {
    "gpt4o-mini": "final_parsing_tokenized",
    "llama":      "llama_parsing_tokenized_clipped",
    "spacy":      "spacy_parsing_tokenized",
}
SPLIT = "val"

print(f"python {sys.version.split()[0]}   cwd {os.getcwd()}\n")

ok = True

sr = f"data/ScanRefer_filtered_{SPLIT}.json"
if os.path.isfile(sr):
    print(f"  OK       {sr}  ({len(json.load(open(sr)))} annotations)")
else:
    ok = False
    print(f"  MISSING  {sr}")

available = {}
for name, folder in PARSERS.items():
    path = f"data_parsing/{folder}/tokenized_parsed_result_{SPLIT}.json"
    if os.path.isfile(path):
        available[name] = folder
        print(f"  OK       {name:12s} {folder}")
    else:
        print(f"  MISSING  {name:12s} {folder}   -> this parser will be skipped")

depth = "outputs/analysis/dep_depth_cache.json"
if os.path.isfile(depth):
    print(f"  OK       {depth}  ({len(json.load(open(depth)))} cached depths)")
else:
    print(f"  ABSENT   {depth}")
    print("           -> experiment 2 computes depths with spaCy on first run (~40 s)")

try:
    import matplotlib
    HAVE_MPL = True
    print(f"  OK       matplotlib {matplotlib.__version__}")
except ImportError:
    HAVE_MPL = False
    print("  ABSENT   matplotlib  -> text charts are shown instead of figures")

print()
if not ok:
    print("STOP: ScanRefer is missing. Fix section 0 before continuing.")
elif len(available) < 2:
    print("WARNING: fewer than two parsers found. The comparison needs at least two.")
else:
    print(f"Ready. {len(available)} parsers: {', '.join(available)}")

PARSE_ARGS = [f"--parse {name}={folder}" for name, folder in available.items()]

### 0d · Display helpers

Small helpers so every result below renders as a real table/chart in the cell output
rather than as raw text. They degrade to plain text when a dependency is missing, so the
notebook never dies on a display problem.

In [ ]:
from IPython.display import HTML, Image, Markdown, display


def show_table(headers, rows, title=None, highlight_col=None, highlight="max"):
    """An HTML table. `highlight_col` shades the best/worst cell in that column."""
    if title:
        display(Markdown(f"**{title}**"))
    values = []
    if highlight_col is not None:
        for r in rows:
            try:
                values.append(float(str(r[highlight_col]).rstrip("%")))
            except (ValueError, TypeError):
                values.append(None)
        clean = [v for v in values if v is not None]
        mark = (max(clean) if highlight == "max" else min(clean)) if clean else None
    else:
        mark = None

    html = ["<table style='border-collapse:collapse;font-size:13px'>"]
    html.append("<tr>" + "".join(
        f"<th style='border-bottom:2px solid #444;padding:4px 10px;text-align:left'>{h}</th>"
        for h in headers) + "</tr>")
    for i, r in enumerate(rows):
        cells_html = []
        for j, c in enumerate(r):
            style = "border-bottom:1px solid #ddd;padding:4px 10px;"
            if mark is not None and j == highlight_col and values[i] == mark:
                style += "background:#FFE8E8;font-weight:bold;"
            cells_html.append(f"<td style='{style}'>{c}</td>")
        html.append("<tr>" + "".join(cells_html) + "</tr>")
    html.append("</table>")
    display(HTML("".join(html)))


def show_figure(path, caption=None):
    """Render a saved PNG inline, or say plainly why it is not there."""
    if os.path.isfile(path):
        if caption:
            display(Markdown(f"*{caption}*"))
        display(Image(filename=path))
    else:
        display(Markdown(f"`{path}` not found — did the cell above run?"))


def text_bars(pairs, unit="%", width=24):
    """A monospace bar chart. The fallback when matplotlib is absent."""
    top = max((v for _, v in pairs), default=0) or 1
    for label, value in pairs:
        filled = int(round(width * value / top))
        print(f"  {label:14s} {'#' * filled}{'.' * (width - filled)} {value:6.1f}{unit}")


print("display helpers ready")

---
## 1 · Field comparison — adjectives and neighbors across parsers

No ground truth exists for these fields, so this is **reference-free** and reports three
complementary signals:

- **coverage** — how often the parser declines the slot entirely (`"not mentioned"`)
- **faithfulness** — fraction of emitted tokens that actually appear in the description
- **agreement** — mean per-annotation Jaccard against the other parsers

On agreement: GPT-4o-mini and LLaMA-3 were built independently, so where they concur is a
defensible reference band, and a parser far outside it is the odd one out.

**Read faithfulness beside coverage, never alone.** A rule-based parser can only copy
tokens verbatim out of the sentence, so it scores near 100% by construction — that is a
property of the method, not evidence of quality. A parser that emitted nothing would score
perfectly.

In [ ]:
!python experiments/analysis/parse_field_comparison.py {" ".join(PARSE_ARGS)} --split {SPLIT}

### 1b · Results as tables — the numbers, rendered

In [ ]:
FC = json.load(open("outputs/analysis/parse_field_comparison/parse_field_comparison.json"))

display(Markdown(f"Split `{FC['split']}` · **{FC['annotations']}** annotations covered by "
                 f"every parser · consensus band: {', '.join(FC['reference'])}"))

for field, blk in FC["fields"].items():
    rows = []
    for name, s in blk["parsers"].items():
        rows.append([
            name,
            f"{100 * s['empty_rate']:.1f}%",
            f"[{100 * s['empty_ci'][0]:.1f}, {100 * s['empty_ci'][1]:.1f}]",
            f"{s['mean_tokens_when_present']:.2f}",
            f"{100 * s['faithfulness']:.1f}%",
            f"{blk['consensus'][name]:.3f}",
        ])
    # Shade the worst coverage -- the column the argument rests on.
    show_table(["parser", "declined", "95% CI", "tokens when present",
                "faithful", "agreement"], rows,
               title=f"{field} — higher `declined` is worse, higher `agreement` is better",
               highlight_col=1, highlight="max")

    show_table(["parser pair", "mean Jaccard"],
               [[k, f"{v:.3f}"] for k, v in blk["pairwise"].items()],
               title=f"{field} — pairwise agreement", highlight_col=1, highlight="min")

### 1c · The saved figure

Three panels: coverage, faithfulness and agreement, for both fields.

In [ ]:
show_figure("outputs/analysis/parse_field_comparison/parse_field_comparison.png",
            "Saved to outputs/analysis/parse_field_comparison/parse_field_comparison.png")

### 1d · Coverage vs agreement, in one chart

The single view that carries the argument: a parser should sit **low and to the right** —
it fills the slot *and* concurs with the others. Anything in the upper-left declines the
slot often *and* disagrees when it does answer.

In [ ]:
if HAVE_MPL:
    import matplotlib.pyplot as plt

    names = list(FC["fields"]["adjectives"]["parsers"])
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for axis, field in zip(axes, FC["fields"]):
        blk = FC["fields"][field]
        for name in names:
            x = blk["consensus"][name]
            y = 100 * blk["parsers"][name]["empty_rate"]
            axis.scatter(x, y, s=160, zorder=3)
            axis.annotate(name, (x, y), textcoords="offset points", xytext=(9, 5),
                          fontsize=9)
        axis.set_xlabel("agreement with the other parsers (mean Jaccard)")
        axis.set_ylabel("declined the slot (%)")
        axis.set_title(field)
        axis.grid(alpha=0.3)
        axis.invert_yaxis()
    figure.suptitle("Better parsers sit lower-right: they answer, and they concur")
    figure.tight_layout()
    plt.show()
else:
    for field, blk in FC["fields"].items():
        print(f"--- {field}: declined the slot (lower is better) ---")
        text_bars([(n, 100 * s["empty_rate"]) for n, s in blk["parsers"].items()])
        print(f"--- {field}: agreement (higher is better) ---")
        text_bars([(n, v) for n, v in blk["consensus"].items()], unit="")
        print()

### 1e · The written verdicts

In [ ]:
display(Markdown(open(
    "outputs/analysis/parse_field_comparison/parse_field_comparison.md").read()))

---
## 2 · The hardest descriptions — parsers head to head

One number per parser over the whole split hides *where* the difference lives. Every
parser handles "there is a black chair" identically; they diverge on long, deeply nested
descriptions — exactly the ones a grounding model needs help with.

So this does both halves at once:

- **the statistic** — the whole split, split into complexity quartiles. This is what makes
  the result generalise instead of being cherry-picked.
- **the anecdote** — the N hardest descriptions with every parser's output side by side.

Complexity is spaCy dependency-tree depth, tie-broken by token count. Depth counts how
deeply the relative clauses nest, which is precisely what a rule chain over noun chunks
has to get right. `--rank-by tokens` or `--rank-by spatial` switch the criterion.

In [ ]:
NUM_CASES = 10

!python experiments/analysis/complex_sentence_showdown.py \
    {" ".join(PARSE_ARGS)} --split {SPLIT} --num {NUM_CASES} --rank-by depth

### 2b · The statistic — declined rate by complexity quartile

In [ ]:
SD = json.load(open(
    "outputs/analysis/complex_sentence_showdown/complex_sentence_showdown.json"))

QUARTILES = ["Q1 simplest", "Q2", "Q3", "Q4 hardest"]
names = list(SD["quartiles"]["adjectives"])

for field, table in SD["quartiles"].items():
    rows = []
    for name in names:
        entries = table[name]
        gap = 100 * (entries[3]["empty_rate"] - entries[0]["empty_rate"])
        rows.append([name] + [f"{100 * e['empty_rate']:.1f}%" for e in entries]
                    + [f"{gap:+.1f} pp"])
    show_table(["parser"] + QUARTILES + ["Q4 − Q1"], rows,
               title=f"{field} — how often the slot is left empty, by description "
                     f"complexity", highlight_col=4, highlight="max")

In [ ]:
if HAVE_MPL:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    x = range(4)
    for axis, (field, table) in zip(axes, SD["quartiles"].items()):
        for name in names:
            y = [100 * e["empty_rate"] for e in table[name]]
            axis.plot(x, y, marker="o", label=name, linewidth=2)
        axis.set_xticks(list(x))
        axis.set_xticklabels(QUARTILES, rotation=12)
        axis.set_ylabel("declined the slot (%)")
        axis.set_title(field)
        axis.grid(alpha=0.3)
        axis.legend(fontsize=8)
    figure.suptitle("Coverage by description complexity — the gap is a level, not a slope")
    figure.tight_layout()
    plt.show()
else:
    for field, table in SD["quartiles"].items():
        print(f"--- {field}: declined rate, simplest -> hardest quartile ---")
        for name in names:
            series = "  ".join(f"{100 * e['empty_rate']:5.1f}%" for e in table[name])
            print(f"  {name:14s} {series}")
        print()

### 2c · The anecdote — the hardest descriptions, side by side

Every selected case, with each parser's three fields. **Red = the parser declined the
slot.** This is the same content as the saved figure, rendered as HTML so it stays
readable and selectable.

In [ ]:
def render_case(case, index):
    header = (f"**{index}. {case['scene_id']} · obj {case['object_id']} · "
              f"ann {case['ann_id']}** — depth {case['depth']}, {case['tokens']} tokens, "
              f"true class `{case['object_name']}`")
    display(Markdown(header))
    display(HTML(
        f"<div style='padding:6px 10px;margin:2px 0 6px 0;border-left:3px solid #888;"
        f"background:#FAFAFA;font-style:italic;font-size:13px'>{case['description']}</div>"))

    html = ["<table style='border-collapse:collapse;font-size:12.5px;width:100%'>"]
    html.append("<tr>" + "".join(
        f"<th style='border-bottom:2px solid #444;padding:4px 8px;text-align:left'>{h}</th>"
        for h in ("parser", "target", "adjectives", "neighbors")) + "</tr>")
    for name, entry in case["parsers"].items():
        cells_html = [f"<td style='border-bottom:1px solid #ddd;padding:4px 8px;"
                      f"font-weight:bold'>{name}</td>",
                      f"<td style='border-bottom:1px solid #ddd;padding:4px 8px;"
                      f"font-family:monospace'>{entry['target']}</td>"]
        for field in ("adjectives", "neighbors"):
            empty = entry[f"{field}_empty"]
            style = ("border-bottom:1px solid #ddd;padding:4px 8px;font-family:monospace;"
                     + ("color:#B00020;background:#FFF4F4;" if empty else ""))
            text = entry[field] + (" ← declined" if empty else "")
            cells_html.append(f"<td style='{style}'>{text}</td>")
        html.append("<tr>" + "".join(cells_html) + "</tr>")
    html.append("</table>")
    display(HTML("".join(html)))


display(Markdown(f"Showing the **{len(SD['cases'])}** hardest of {SD['annotations']} "
                 f"descriptions, ranked by `{SD['rank_by']}`."))
for n, case in enumerate(SD["cases"], 1):
    render_case(case, n)

### 2d · The saved figure

The same comparison as a single publication-ready image: the quartile chart on top, one
row per case below.

In [ ]:
show_figure("outputs/analysis/complex_sentence_showdown/complex_sentence_showdown.png",
            "Saved to outputs/analysis/complex_sentence_showdown/"
            "complex_sentence_showdown.png")

### 2e · The written verdicts

In [ ]:
report = open(
    "outputs/analysis/complex_sentence_showdown/complex_sentence_showdown.md").read()
# The per-case tables are already rendered above; show the statistical half only.
display(Markdown(report.split("## The hardest descriptions")[0]))

---
## 3 · What was produced, and what it does *not* say

In [ ]:
for folder in ("parse_field_comparison", "complex_sentence_showdown"):
    base = os.path.join("outputs", "analysis", folder)
    display(Markdown(f"**`{base}/`**"))
    if not os.path.isdir(base):
        display(Markdown("_(not produced — did the cell above fail?)_"))
        continue
    rows = [[name, f"{os.path.getsize(os.path.join(base, name)) / 1024.0:.1f} KB"]
            for name in sorted(os.listdir(base))]
    show_table(["file", "size"], rows)

### Boundaries — worth stating in the paper

**These measure parse quality, not grounding accuracy.** No model is run and no
`predictions.p` is read. The causal question — does a worse parse produce a worse box —
belongs to the corruption experiment (`run_parse_corruption.py` →
`parse_error_propagation.py`), which needs a GPU and has **not been run**:
`outputs/<run>/corruption/` is empty.

**Agreement is not correctness.** Two parsers can agree and both be wrong. The LLM
consensus is a reference *band*, not ground truth. The only way to get true ground truth
for these two fields is to fill in the annotation sheets under
`outputs/analysis/annotation/` — currently 0 of 200 rows for every parser.

**Report the honest negatives too**, because a reviewer will find them otherwise:

- spaCy does **not** degrade faster with complexity — its slope is flatter than the LLMs'.
  The damning number is the *level*, not the slope: it starts about twice as bad and stays
  there.
- Spatial-cue recall barely separates the parsers, and spaCy emits *more* neighbor phrases
  than GPT. The story is coverage and agreement, not recall.

### Related experiments

| Script | Needs | What it adds |
|---|---|---|
| `experiments/ablation/parsers/eval_parser_target_accuracy.py` | CPU | the `target` field, against real ground truth |
| `experiments/analysis/parse_quality_split.py` | `predictions.p` | association between parse errors and grounding |
| `experiments/ablation/runners/run_parse_corruption.py` | **GPU** | the causal test — currently unrun |